# Exploration of Credit Card Transactions Dataset

This notebook performs an initial exploratory data analysis (EDA) of the raw credit card transactions dataset.

Goals:
- Understand schema, data types, and basic distributions
- Inspect fraud vs non-fraud behaviour
- Identify potential data quality issues (nulls, duplicates, invalid values)
- Derive insights that will inform Silver and Gold layer design


In [6]:
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Adjust path if needed
RAW_PATH = "../data/raw/creditcard.csv"

spark = (
    SparkSession.builder
    .appName("Exploration")
    .getOrCreate()
)


In [7]:
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_PATH)
)

df_raw.printSchema()
df_raw.show(5)
print("Row count:", df_raw.count())


root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

In [8]:
# Column list
df_raw.columns

['Time',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'Amount',
 'Class']

In [9]:
# Basic describe for numeric columns
df_raw.describe().show()

+-------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+
|summary|             Time|                  V1|                  V2|                  V3|                  V4|                  V5|                  V6|                  V7|                  V8|                  V9|                 V10|                 V11|                 V12|                 V13|                 V14|                 V15|   

In [10]:
null_counts = df_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
])

null_counts.show(truncate=False)


+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|Time|V1 |V2 |V3 |V4 |V5 |V6 |V7 |V8 |V9 |V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Amount|Class|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|0   |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0     |0    |
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+



In [11]:
total_rows = df_raw.count()
distinct_rows = df_raw.dropDuplicates().count()

print("Total rows:    ", total_rows)
print("Distinct rows: ", distinct_rows)
print("Duplicates:    ", total_rows - distinct_rows)


Total rows:     284807
Distinct rows:  283726
Duplicates:     1081


In [12]:
df_raw.select("Amount", "Time", "Class").describe().show()


+-------+------------------+-----------------+--------------------+
|summary|            Amount|             Time|               Class|
+-------+------------------+-----------------+--------------------+
|  count|            284807|           284807|              284807|
|   mean| 88.34961925094233|94813.85957508067|0.001727485630620034|
| stddev|250.12010924018833|47488.14595456595| 0.04152718963546499|
|    min|               0.0|              0.0|                   0|
|    max|          25691.16|         172792.0|                   1|
+-------+------------------+-----------------+--------------------+



In [13]:
# Negative or zero amounts
df_raw.filter(F.col("Amount") < 0).count(), df_raw.filter(F.col("Amount") == 0).count()


(0, 1825)

In [14]:
# Time range
df_raw.select(F.min("Time").alias("min_time"), F.max("Time").alias("max_time")).show()


+--------+--------+
|min_time|max_time|
+--------+--------+
|     0.0|172792.0|
+--------+--------+



In [15]:
# Class distribution
class_counts = df_raw.groupBy("Class").count().orderBy("Class")
class_counts.show()

total = df_raw.count()
class_counts.withColumn("rate", F.col("count") / F.lit(total)).show()


+-----+------+
|Class| count|
+-----+------+
|    0|284315|
|    1|   492|
+-----+------+

+-----+------+--------------------+
|Class| count|                rate|
+-----+------+--------------------+
|    0|284315|  0.9982725143693799|
|    1|   492|0.001727485630620034|
+-----+------+--------------------+



In [16]:
amount_buckets = (
    df_raw.withColumn(
        "amount_bucket",
        F.when(F.col("Amount") < 10, "0-10")
         .when((F.col("Amount") >= 10) & (F.col("Amount") < 50), "10-50")
         .when((F.col("Amount") >= 50) & (F.col("Amount") < 100), "50-100")
         .when((F.col("Amount") >= 100) & (F.col("Amount") < 500), "100-500")
         .when((F.col("Amount") >= 500) & (F.col("Amount") < 1000), "500-1000")
         .otherwise("1000+")
    )
    .groupBy("amount_bucket")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("Class").alias("total_frauds"),
        (F.sum("Class") / F.count("*")).alias("fraud_rate")
    )
    .orderBy("amount_bucket")
)

amount_buckets.show(truncate=False)


+-------------+------------------+------------+---------------------+
|amount_bucket|total_transactions|total_frauds|fraud_rate           |
+-------------+------------------+------------+---------------------+
|0-10         |97314             |249         |0.0025587274184598312|
|10-50        |92390             |56          |6.061262041346466E-4 |
|100-500      |47893             |95          |0.0019835884158436513|
|1000+        |3069              |9           |0.002932551319648094 |
|50-100       |37718             |57          |0.001511214804602577 |
|500-1000     |6423              |26          |0.0040479526700918575|
+-------------+------------------+------------+---------------------+



In [17]:
df_hour = df_raw.withColumn("hour", (F.col("Time") / 3600).cast("int"))

hourly = (
    df_hour.groupBy("hour")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("Class").alias("total_frauds"),
        (F.sum("Class") / F.count("*")).alias("fraud_rate")
    )
    .orderBy("hour")
)

hourly.show(24, truncate=False)


+----+------------------+------------+---------------------+
|hour|total_transactions|total_frauds|fraud_rate           |
+----+------------------+------------+---------------------+
|0   |3963              |2           |5.046681806712087E-4 |
|1   |2217              |2           |9.021199819576004E-4 |
|2   |1576              |21          |0.0133248730964467   |
|3   |1821              |13          |0.007138934651290499 |
|4   |1082              |6           |0.005545286506469501 |
|5   |1681              |11          |0.006543723973825104 |
|6   |1831              |3           |0.0016384489350081922|
|7   |3368              |23          |0.006828978622327791 |
|8   |5179              |5           |9.654373431164317E-4 |
|9   |7878              |15          |0.001904036557501904 |
|10  |8288              |2           |2.4131274131274132E-4|
|11  |8517              |43          |0.00504872607725725  |
|12  |7732              |9           |0.0011639937920331091|
|13  |7585              

In [22]:
# Correlations between Class and Amount, V1 - V28 
target_col = "Class"

feature_cols = [
    "Amount",
    "V1", "V2", "V3", "V4", "V5", "V6", "V7", "V8", "V9",
    "V10", "V11", "V12", "V13", "V14", "V15", "V16", "V17",
    "V18", "V19", "V20", "V21", "V22", "V23", "V24", "V25",
    "V26", "V27", "V28"
]

correlations = []

for col in feature_cols:
    corr = df_raw.stat.corr(col, target_col)
    correlations.append((col, corr))
#    print(f"Correlation {col} – {target_col}: {corr}")

correlations_sorted = sorted(correlations, key=lambda x: abs(x[1]), reverse=True)

print("\nSorted by |correlation|:")
for col, corr in correlations_sorted:
    print(f"{col:6s}  ->  corr = {corr:.6f}")


Sorted by |correlation|:
V17     ->  corr = -0.326481
V14     ->  corr = -0.302544
V12     ->  corr = -0.260593
V10     ->  corr = -0.216883
V16     ->  corr = -0.196539
V3      ->  corr = -0.192961
V7      ->  corr = -0.187257
V11     ->  corr = 0.154876
V4      ->  corr = 0.133447
V18     ->  corr = -0.111485
V1      ->  corr = -0.101347
V9      ->  corr = -0.097733
V5      ->  corr = -0.094974
V2      ->  corr = 0.091289
V6      ->  corr = -0.043643
V21     ->  corr = 0.040413
V19     ->  corr = 0.034783
V20     ->  corr = 0.020090
V8      ->  corr = 0.019875
V27     ->  corr = 0.017580
V28     ->  corr = 0.009536
V24     ->  corr = -0.007221
Amount  ->  corr = 0.005632
V13     ->  corr = -0.004570
V26     ->  corr = 0.004455
V15     ->  corr = -0.004223
V25     ->  corr = 0.003308
V23     ->  corr = -0.002685
V22     ->  corr = 0.000805


## Conclusions from Exploration

- The dataset contains 284,807 credit card transactions with a very strong class imbalance: fraud cases represent approximately 0.17% of all records.
- No missing values were detected in any column, which simplifies downstream cleaning.
- A total of **1,081 duplicate rows** were found in the raw dataset. These will need to be removed or handled during the Silver layer processing.
- The `Amount` column shows a highly skewed distribution: most transactions are small, with a long tail of larger amounts. Fraud does not appear to be concentrated in either very small or very large amounts.
- The `Time` column ranges from 0 to ~172,792 seconds (about 48 hours), confirming that the dataset covers two days of transactions.
- Fraud transactions do not show a strong linear relationship with the transaction amount (correlation close to zero).
- PCA components `V1–V28` exhibit varying degrees of correlation with `Class`, with some components showing stronger relationships than others. These components carry most of the predictive signal in the dataset.
- No invalid values were detected (e.g., negative amounts, invalid class labels), but further validation will be performed in the Data Quality Checks.

### How these findings guide the next steps

- **Data Quality Checks (dq_checks.py):**  
  Will formally validate schema, ranges, duplicates, and potential anomalies identified during exploration.

- **Silver Layer (etl_silver.py):**  
  Will apply standardization, type enforcement, and remove the 1,081 duplicate rows identified during exploration.

- **Gold Layer (etl_gold.py):**  
  Will compute business‑ready aggregates such as fraud statistics, hourly patterns, and amount‑based metrics, informed by the patterns observed in the exploration.
in the exploration.
n the exploration.
 etl_gold.py
